## <span style="color:Purple">You are allowed to use core Python's built in modules/packages/libraries, NumPy, Pandas, scikit-learn, matplotlib, and Seaborn.</span>

### <span style="color:Red">Examples and Resources for this assignment:</span>
<ul>
    <li><span style="color:Red">Chapters 3, 4, 5, 6, 7, 8, and 9 from <a href="https://docs.python.org/3/tutorial/index.html">The Python Tutorial</a></span></li>
    <li><span style="color:Red">Chapter 2 from <a href="https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html">Introduction to NumPy</a></span></li>
    <li><span style="color:Red">Chapter 3 from <a href="https://jakevdp.github.io/PythonDataScienceHandbook/02.00-introduction-to-numpy.html">Data Manipulation with Pandas</a></span></li>
    <li><span style="color:Red">Chapter 2 from <a href="https://github.com/ageron/handson-ml3/blob/main/02_end_to_end_machine_learning_project.ipynb">End to End Machine Learning Project</a></span></li>
    <li><span style="color:Red">Chapter 4 from <a href="https://github.com/ageron/handson-ml3/blob/main/04_training_linear_models.ipynb">Training Linear Models</a></span></li>
</ul>

### <span style="color:Green">Context</span>
The attached CSV file contains daily weather data for the year 2024 for the city of Toronto, Ontario, Canada. It includes key meteorological variables such as:

<ul>
    <li><span><b>Date:</b> The recorded date of the weather data.</span></li>
    <li><span><b>Mean Temp ($^{\circ}$C):</b> The average daily temperature in degrees Celsius.</span></li>
    <li><span><b>Total Precip (mm):</b> The total daily precipitation (rain and melted snow) in millimeters.</span></li>
    <li><span><b>Snowfall (cm):</b> The amount of snowfall in centimeters.</span></li>
    <li><span><b>Wind Speed (km/h):</b> The recorded daily wind speed in kilometers per hour.</span></li>
</ul>

The following <a href="https://climate.weather.gc.ca/glossary_e.html">link</a> might be useful for the description of the features. With this dataset predict the mean temperature of a day.

# <span style="color:Green">P1: Load the dataset.</span>

In [30]:
# Import the libraries needed for data analysis, plots, and machine learning.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Keep the style and split reproducible every time the notebook is run.
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# Load the CSV file and convert the date column to datetime format.
weather_df = pd.read_csv("out.csv")
weather_df["Date/Time"] = pd.to_datetime(weather_df["Date/Time"])

# Detect the important temperature columns by name.
target_column = next(col for col in weather_df.columns if "Mean Temp" in col and "Flag" not in col)
max_temp_column = next(col for col in weather_df.columns if "Max Temp" in col and "Flag" not in col)
min_temp_column = next(col for col in weather_df.columns if "Min Temp" in col and "Flag" not in col)
heat_degree_column = next(col for col in weather_df.columns if "Heat Deg Days" in col and "Flag" not in col)
cool_degree_column = next(col for col in weather_df.columns if "Cool Deg Days" in col and "Flag" not in col)

# Preview the dataset to confirm it loaded correctly.
print("First five rows:")
display(weather_df.head())
print(f"Date range: {weather_df['Date/Time'].min().date()} to {weather_df['Date/Time'].max().date()}")


First five rows:


,Longitude (x),Latitude (y),Station Name,Climate ID,Date/Time,Year,Month,Day,Data Quality,Max Temp (°C),...,Total Snow (cm),Total Snow Flag,Total Precip (mm),Total Precip Flag,Snow on Grnd (cm),Snow on Grnd Flag,Dir of Max Gust (10s deg),Dir of Max Gust Flag,Spd of Max Gust (km/h),Spd of Max Gust Flag
0,-79.4,43.67,TORONTO CITY,6158355,2021-01-01,2021,1,1,NaN,2.5,...,NaN,NaN,6.8,NaN,NaN,NaN,NaN,M,NaN,M
1,-79.4,43.67,TORONTO CITY,6158355,2021-01-02,2021,1,2,NaN,2.2,...,NaN,NaN,10.8,NaN,0.0,NaN,NaN,M,NaN,M
2,-79.4,43.67,TORONTO CITY,6158355,2021-01-03,2021,1,3,NaN,2.1,...,NaN,NaN,1.7,NaN,0.0,NaN,NaN,M,NaN,M
3,-79.4,43.67,TORONTO CITY,6158355,2021-01-04,2021,1,4,NaN,1.7,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,M,NaN,M
4,-79.4,43.67,TORONTO CITY,6158355,2021-01-05,2021,1,5,NaN,1.6,...,NaN,NaN,0.4,NaN,NaN,NaN,NaN,M,NaN,M


Date range: 2021-01-01 to 2025-12-31


# <span style="color:Green">P2: Print a quick description of the data.</span>

In [31]:
# Show the dataframe structure, types, and non-null counts.
print("Dataset info:")
weather_df.info()

# Print the shape and target name for quick reference.
print(f"\nShape: {weather_df.shape[0]} rows x {weather_df.shape[1]} columns")
print(f"Target column: {target_column}")

# Sort missing values so the biggest data-quality issues appear first.
missing_summary = weather_df.isna().sum().sort_values(ascending=False)
print("\nColumns with missing values:")
display(missing_summary[missing_summary > 0])


Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 1826 entries, 0 to 1825
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Longitude (x)              1826 non-null   float64       
 1   Latitude (y)               1826 non-null   float64       
 2   Station Name               1826 non-null   str           
 3   Climate ID                 1826 non-null   int64         
 4   Date/Time                  1826 non-null   datetime64[us]
 5   Year                       1826 non-null   int64         
 6   Month                      1826 non-null   int64         
 7   Day                        1826 non-null   int64         
 8   Data Quality               0 non-null      float64       
 9   Max Temp (°C)              1818 non-null   float64       
 10  Max Temp Flag              8 non-null      str           
 11  Min Temp (°C)              1818 non-null   float64       
 12  Min

Data Quality                 1826
Heat Deg Days Flag           1818
Cool Deg Days Flag           1818
Max Temp Flag                1818
Min Temp Flag                1818
Mean Temp Flag               1818
Snow on Grnd Flag            1801
Total Rain Flag              1795
Total Snow Flag              1783
Total Precip Flag            1750
Spd of Max Gust (km/h)       1540
Dir of Max Gust (10s deg)    1540
Total Rain (mm)              1462
Total Snow (cm)              1462
Snow on Grnd (cm)            1415
Spd of Max Gust Flag          286
Dir of Max Gust Flag          286
Total Precip (mm)               8
Heat Deg Days (°C)              8
Cool Deg Days (°C)              8
Mean Temp (°C)                  8
Min Temp (°C)                   8
Max Temp (°C)                   8
dtype: int64

# <span style="color:Green">P3: Display a summary of the numerical attributes.</span>

In [32]:
# Use numeric columns only for the descriptive statistics table.
numeric_columns = [
    col for col in weather_df.select_dtypes(include=np.number).columns
    if weather_df[col].notna().any()
]

# Add missing-value counts to the usual summary statistics.
numeric_summary = weather_df[numeric_columns].describe().T
numeric_summary["missing_values"] = weather_df[numeric_columns].isna().sum()
display(numeric_summary)


,count,mean,std,min,25%,50%,75%,max,missing_values
Longitude (x),1826.0,-7.944597e+01,0.092006,-79.63,-79.40,-79.40,-79.40,-79.40,0
Latitude (y),1826.0,4.367200e+01,0.004000,43.67,43.67,43.67,43.67,43.68,0
Climate ID,1826.0,6.158430e+06,150.410295,6158355.00,6158355.00,6158355.00,6158355.00,6158731.00,0
Year,1826.0,2.023001e+03,1.414407,2021.00,2022.00,2023.00,2024.00,2025.00,0
Month,1826.0,6.523549e+00,3.449478,1.00,4.00,7.00,10.00,12.00,0
Day,1826.0,1.572782e+01,8.801735,1.00,8.00,16.00,23.00,31.00,0
Max Temp (°C),1818.0,1.435512e+01,10.642347,-12.60,5.10,14.55,24.00,36.00,8
Min Temp (°C),1818.0,6.413091e+00,9.555579,-20.50,-0.50,6.40,15.00,24.80,8
Mean Temp (°C),1818.0,1.038762e+01,9.988180,-16.50,2.30,10.30,19.50,30.20,8
Heat Deg Days (°C),1818.0,8.826403e+00,8.549858,0.00,0.00,7.70,15.70,34.50,8


# <span style="color:Green">P4: Plot a histogram for each numerical attribute.</span>

# <span style="color:Green">P5: Compute the standard correlation coefficient (also called Pearson’s r) between every pair of attributes.</span>

# <span style="color:Green">P6: Set the missing values to the median (if any).</span>

# <span style="color:Green">P7: Split your dataset into training set (80%) and test set (20%).</span>

# <span style="color:Green">P8: Use LinearRegression, DecisionTreeRegressor, and RandomForestRegressor to train your model. (apply 5-fold cross validation). For each regression model calculate and print RMSE score. </span>

# <span style="color:Green">P9: Calculate and print RMSE score for the test set. </span>

# <span style="color:Green">P10: Predict the mean temperature of a date and compare it with the actual mean temperature.</span>